# LM5176 EN/UVLO Latch — Design Analysis

**Project:** USB-C PD → 12 V Fan-Array Controller  
**Schematic:** `en_uvlo_lockoutv2.asc`  **Spec source of truth:** `constants.py`

## Problem
The LM5176 EN/UVLO pin is normally driven by a plain VIN resistor divider. On a brief
VIN dropout the part can re-enable while the output capacitors are still charged
(a **pre-bias** condition), which stresses the converter at restart.

## Goal of the latch
Add a discrete latch that, on a VIN dropout, **pulls EN/UVLO low and holds it low until
VOUT has decayed below a safe level**, ignoring any VIN re-application during that window.
This forces a clean, fully-discharged restart instead of a pre-biased one.

## What this notebook does
1. Computes the LM5176 internal UVLO thresholds and checks them against the design targets.
2. Shows *why* an external latch is justified (the divider alone is over-constrained).
3. States the four operating regions the latch must satisfy, with consistent notation.
4. Explores the latch resistor design space (vectorised, one plot per sub-network).


## 0. Reference designators and notation

All symbols below map to **`en_uvlo_lockoutv2.asc`** as drawn. The original notebook used
three different naming schemes (e.g. the EN/UVLO divider was called `R8/R9` in one cell and
`R1/R2` in others); everything here uses the single mapping in this table.

| Schematic | Value (as drawn) | Role | Symbol used here |
|-----------|------------------|------|------------------|
| R2 | 300 k | EN/UVLO divider, top (VIN→EN)   | `R_uv_top` |
| R1 | 100 k | EN/UVLO divider, bottom (EN→GND) | `R_uv_bot` |
| R4 | 4.9 k | VIN-sense divider, top (VIN→Q2 base) | `R_sns_top` |
| R3 | 1 k   | VIN-sense divider, bottom (→GND)     | `R_sns_bot` |
| R6 | 30 M  | VOUT→latch feedback (high side)  | `R_fb_hi` |
| R5 | 200 k | latch node→GND (low side)        | `R_fb_lo` |
| R8 | 300 k | latch-core coupling / Q4 drive   | `R_core_a` |
| R7 | 3 M   | latch-core coupling              | `R_core_b` |
| Q2 | 2N3904 (NPN) | **VIN-dropout sensor** | — |
| Q3 + Q4 | 2N3904 / 2N3906 | **regenerative latch core (SCR)** | — |
| Q1 | 2N3904 (NPN) | **EN/UVLO pull-down** (latch output) | — |

> ⚠️ **Verify before trusting the region search (§4):** transistor pin assignments and the
> exact node a given resistor lands on were read from the `.asc` wiring, not from a simulated
> netlist. The threshold math in §2–§3 depends only on the divider values and is solid; the
> latch-core inequalities in §4 reproduce *your* intended constraints and should be confirmed
> against an LTspice `.op`/`.tran` run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# LM5176 EN/UVLO specs — pulled from constants.py (single source of truth).
# NOTE: the original cell referenced IC_LM5176.V_EN_OP_TYP / I_EN_STBY_NOM / I_HYS_TYP,
#       which do NOT exist and raise AttributeError. Correct names are below.
from constants import IC_LM5176 as IC, DESIGN_TARGETS as T

V_EN   = IC.V_EN_NOM      # 1.22 V  enable comparator threshold (typ)
V_EN_MIN, V_EN_MAX = IC.V_EN_MIN, IC.V_EN_MAX
I_HYS  = IC.I_HYS_NOM     # 3.15 uA hysteresis current (sourced out of pin when enabled)
I_STBY = IC.I_STBY_NOM    # 2.0  uA standby pin current

# First-order BJT model (used by the latch region analysis in section 4)
VBE_ON   = 0.65   # base-emitter turn-on (active)
VBE_SAT  = 0.75   # base-emitter at hard saturation
VBE_OFF  = 0.50   # below this -> transistor effectively off
VCE_SAT  = 0.10   # collector-emitter when saturated (typ)
VCE_SAT_MAX = 0.20
BETA_FORCED = 10  # forced-beta design margin: keep Ic/Ib <= this for guaranteed saturation

# Operating envelope
VOUT_NOM     = 12.0
VIN_PD_MAX   = 22.0   # 20 V PD * 1.1 tolerance
print(f"V_EN={V_EN} V  (min {V_EN_MIN}, max {V_EN_MAX}) | I_HYS={I_HYS*1e6:.2f} uA | I_STBY={I_STBY*1e6:.2f} uA")


## 1. LM5176 EN/UVLO threshold equations

For a top/bottom divider (`R_uv_top` from VIN, `R_uv_bot` to GND) feeding the EN/UVLO pin:

$$ V_{IN,\text{rise}} = V_{EN}\left(1 + \frac{R_{uv,top}}{R_{uv,bot}}\right) $$

When the part enables, the internal hysteresis current $I_{HYS}$ is sourced **out of the pin**
through the top resistor, so VIN can fall further before the pin drops back below $V_{EN}$:

$$ V_{IN,\text{fall}} = V_{IN,\text{rise}} - I_{HYS}\,R_{uv,top}
   \qquad\Rightarrow\qquad V_{HYS} = I_{HYS}\,R_{uv,top} $$

The standby current $I_{STBY}$ only matters when it stops being negligible against the divider
current ($\approx V_{EN}/R_{uv,bot}$); it is reported below as a sanity ratio rather than folded
into the threshold (the original cell subtracted $I_{STBY}R_{top}$ from the *rising* threshold,
which is the wrong term — hysteresis comes from $I_{HYS}$, not $I_{STBY}$).


In [ ]:
def uvlo(r_top, r_bot):
    v_rise = V_EN * (1 + r_top / r_bot)
    v_fall = v_rise - I_HYS * r_top
    return v_rise, v_fall

# As-built divider (R2 = 300k top, R1 = 100k bottom)
R_uv_top, R_uv_bot = 300e3, 100e3
v_rise, v_fall = uvlo(R_uv_top, R_uv_bot)
v_hys = v_rise - v_fall
i_div_at_thresh = V_EN / R_uv_bot
p_div_max = VIN_PD_MAX**2 / (R_uv_top + R_uv_bot)

print(f"As-built divider: R_uv_top={R_uv_top/1e3:.0f}k  R_uv_bot={R_uv_bot/1e3:.0f}k  (ratio {R_uv_bot/(R_uv_top+R_uv_bot):.3f})")
print(f"  VIN rising  enable  : {v_rise:.3f} V")
print(f"  VIN falling disable : {v_fall:.3f} V")
print(f"  Hysteresis          : {v_hys:.3f} V")
print(f"  Divider I @ thresh  : {i_div_at_thresh*1e6:.2f} uA  (vs I_STBY {I_STBY*1e6:.1f} uA, I_HYS {I_HYS*1e6:.2f} uA)")
print(f"  Divider power @ {VIN_PD_MAX:.0f} V: {p_div_max*1e6:.0f} uW")

# --- Check against DESIGN_TARGETS ---
checks = [
    ("Enable at/below V_USB_MIN",  v_rise <= T.V_USB_MIN,        f"{v_rise:.2f} <= {T.V_USB_MIN}"),
    ("Enable at/above safe floor", v_rise >= T.V_ON_SAFE_FLOOR,  f"{v_rise:.2f} >= {T.V_ON_SAFE_FLOOR}"),
    ("Disable above V_OFF_MIN",    v_fall >= T.V_OFF_MIN,        f"{v_fall:.2f} >= {T.V_OFF_MIN}"),
    ("Hysteresis >= V_HYS_MIN",    v_hys  >= T.V_HYS_MIN,        f"{v_hys:.2f} >= {T.V_HYS_MIN}"),
    ("Divider power <= budget",    p_div_max*1e6 <= T.MAX_DIVIDER_UW, f"{p_div_max*1e6:.0f} <= {T.MAX_DIVIDER_UW} uW"),
]
print("\n  TARGET CHECK")
for name, ok, detail in checks:
    print(f"   [{'PASS' if ok else 'FAIL'}] {name:28s} ({detail})")


## 2. Finding: the divider alone is over-constrained — this is *why* the latch exists

Two targets pull the top resistor in opposite directions:

* **Hysteresis ceiling.** Keeping $V_{IN,fall}\ge 3.6$ V while $V_{IN,rise}\le 4.5$ V means
  $V_{HYS}=I_{HYS}R_{uv,top}\le 0.9$ V, i.e. $R_{uv,top}\lesssim 286\,\text{k}$.
* **Power budget.** Staying under $500\,\mu W$ at 22 V needs
  $R_{uv,top}+R_{uv,bot}\ge 968\,\text{k}$.

A small enough top resistor for the hysteresis target forces a low total resistance, which blows
the power budget by 2–7×. You **cannot** get a tight, low-power turn-off purely from the IC's
internal hysteresis at this $I_{HYS}$.

**Implication:** size the divider for the *rising* threshold and power only, accept whatever
internal $V_{IN,fall}$ falls out, and let the **external latch define the off-condition** (and
ride through pre-bias). The cell below shows the trade-off explicitly.


In [ ]:
# Sweep top resistor; for each, set bottom for a centred rising threshold of 4.15 V,
# then report the resulting falling threshold, hysteresis and power.
V_RISE_TARGET = 4.15
rt = np.array([100e3, 150e3, 200e3, 250e3, 286e3, 470e3, 750e3, 1e6])
rb = rt / (V_RISE_TARGET / V_EN - 1)
vr, vf = uvlo(rt, rb)
p22 = VIN_PD_MAX**2 / (rt + rb) * 1e6

print(f"Centred rising threshold = {V_RISE_TARGET} V\n")

import pandas as pd
df = pd.DataFrame({
    "R_top (k)":  (rt/1e3).round(0),
    "R_bot (k)":  (rb/1e3).round(1),
    "V_rise":     vr.round(2),
    "V_fall":     vf.round(2),
    "V_hys":      (vr-vf).round(2),
    "P@22V (uW)": p22.round(0),
    "fall>=3.6":  vf >= T.V_OFF_MIN,
    "P<=500uW":   p22 <= T.MAX_DIVIDER_UW,
})
print(df.to_string(index=False))
print("\nNo row satisfies BOTH 'fall>=3.6' and 'P<=500uW' -> external latch sets the off-point.")


## 3. The four operating regions the latch must satisfy

`VB3` is the latch-core base node (Q3/Q4); `Q1` is the EN/UVLO pull-down; `Q2` is the
VIN-dropout sensor driven by the `R_sns_top`/`R_sns_bot` divider.

**(a) Run / steady state — must NOT latch.**  `VIN` high and `VOUT` high. Q2 is on, holding
the latch core off so `VB3` stays below `VBE_OFF`:
$$ V_{B3}\big|_{\text{run}} < V_{BE,off} $$

**(b) SET on VIN dropout.**  `VIN` falling, `VOUT` still high. When VIN drops far enough that
Q2 turns **off**, the feedback from VOUT pulls `VB3` up and the core latches. The Q2 release
point is:
$$ V_{IN,set} \approx V_{BE,on}\,\frac{R_{sns,top}+R_{sns,bot}}{R_{sns,bot}} $$
This **must sit at/above the IC's own $V_{IN,fall}$** so the latch grabs control before the IC
self-disables (the `LATCH_TO_IC_OFF_GAP` margin).

**(c) HOLD with VIN back high.**  `VIN` high again, `VOUT` still high. Q2 turns on and fights
the latch, but the VOUT feedback into `VB3` must still win:
$$ I_{fb}(V_{OUT}) \;>\; I_{sink,Q2} + I_{B,Q3} + I_{B,Q4} $$

**(d) RESET when VOUT decays.**  `VIN` high, `VOUT` low. As VOUT falls the feedback weakens;
below the release point the sink wins, `VB3` drops below `VBE_OFF`, the core releases and EN
recovers:
$$ I_{fb}(V_{OUT}) \;<\; I_{sink} \quad\text{for } V_{OUT} < V_{OUT,release} $$


In [ ]:
# ---- Region (b): VIN-dropout sensor (Q2) trip point ----
# Q2 turns OFF (latch is allowed to SET) when its base divider can no longer hold VBE_ON.
R_sns_top, R_sns_bot = 4.9e3, 1e3          # R4, R3
def vin_set(vbe):  return vbe * (R_sns_top + R_sns_bot) / R_sns_bot

print("VIN-dropout SET threshold (Q2 release):")
for vbe in (VBE_OFF, VBE_ON, VBE_SAT):
    print(f"   VBE={vbe:.2f} V -> VIN_set = {vin_set(vbe):.2f} V")
print(f"\nIC self-disable (V_fall, as-built divider) = {v_fall:.2f} V")
gap = vin_set(VBE_ON) - v_fall
print(f"Latch-vs-IC margin at VBE_ON: {gap:+.2f} V "
      f"({'OK - latch sets first' if gap>0 else 'RISK - IC may disable before latch sets'})")
print("Note: trip is VBE-sensitive (temperature/part spread). Design R_sns for margin >= "
      f"{T.HYS_MIN_GAP} V above V_fall across VBE spread.")


In [ ]:
# ---- Region (a)+(b): Q2 switching design space  (cleaned from original cell 8) ----
# Role-based names. Sweep the sense divider and require:
#   c_release : at VIN_low, Q2 base < VBE_OFF  (Q2 off -> latch may set)
#   c_run     : at VIN_high, Q2 has enough base drive to saturate (latch held off)
#
# CORRECTION: the original evaluated "Q2 must hold off" at 3.8 V and "Q2 must release"
# at ~3.6 V. Those windows overlap and have NO solution. "Hold off" is a NORMAL-OPERATION
# condition, so evaluate it at the operating minimum (USB 5 V); release is below the trip.
from constants import USB
V_IN_HIGH = USB.V_MIN            # 5 V operating: Q2 must be solidly ON (latch held off)
V_IN_LOW  = 3.0                  # below the dropout trip: Q2 OFF (latch free to set)
R_core_b  = 3e6                  # R7 (latch-core), drives the base-current requirement
IB_REQ    = (VOUT_NOM - VBE_OFF) / R_core_b / BETA_FORCED

r = np.logspace(2, 5, 500)        # 100 Ohm .. 100 k
Rbot, Rtop = np.meshgrid(r, r)    # Rbot=R_sns_bot, Rtop=R_sns_top

v_base_low = V_IN_LOW * (Rbot / (Rbot + Rtop))
c_release  = v_base_low < VBE_OFF
ib_high    = (V_IN_HIGH - VBE_ON) / Rtop - VBE_ON / Rbot
c_run      = ib_high > IB_REQ
valid      = c_release & c_run

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(Rbot, Rtop, valid, levels=[0.5, 1.5], colors=['#2ca02c'], alpha=0.30)
ax.contour(Rbot, Rtop, v_base_low, levels=[VBE_OFF], colors='C3', linestyles='--')
ax.contour(Rbot, Rtop, ib_high,    levels=[IB_REQ], colors='C0')
ax.plot(R_sns_bot, R_sns_top, 'k*', ms=14, label='as-built (R3,R4)')
ax.set(xscale='log', yscale='log',
       xlabel=r'$R_{sns,bot}$ (R3) [$\Omega$]', ylabel=r'$R_{sns,top}$ (R4) [$\Omega$]',
       title='Q2 sense divider — release (red --) vs run-hold (blue), valid = green')
ax.grid(True, which='both', alpha=0.2); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()


In [ ]:
# ---- Region (c)+(d): latch feedback R_fb_lo / R_fb_hi design space (cleaned from cell 9) ----
# c_hold  : with VOUT still high near release, feedback beats the sink + base demand (stay latched)
# c_reset : once VOUT is low, the node collapses below VBE_OFF (release)
V_OUT_RELEASE = 2.0
R_core_a, R_core_b = 300e3, 3e6      # R8, R7

r = np.logspace(3, 7, 500)           # 1k .. 10M
R_lo, R_hi = np.meshgrid(r, r)       # R_lo=R_fb_lo (R5), R_hi=R_fb_hi (R6)

i_sink    = (VBE_SAT - VCE_SAT) / R_lo
i_fb_hold = (V_OUT_RELEASE - VBE_SAT - VCE_SAT)/R_core_a + (VOUT_NOM - VBE_SAT)/R_hi
ib_demand = (VIN_PD_MAX/R_core_a)/BETA_FORCED + (VOUT_NOM/(R_core_b))/BETA_FORCED
c_hold    = i_fb_hold > (i_sink + ib_demand)

# reset: once VOUT has decayed to the release level the high-side source must be
# beaten by the low-side sink so the node collapses below VBE_OFF.
# CORRECTION: reset is a LOW-VOUT event -> evaluate the source term at V_OUT_RELEASE,
# not VOUT_NOM (the original cell mixed nominal and release inside the reset terms).
i_src_reset  = (V_OUT_RELEASE - VBE_OFF) / R_hi
i_sink_reset = (VBE_OFF - VCE_SAT) / R_lo
c_reset      = i_sink_reset > i_src_reset

valid = c_hold & c_reset
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(R_lo, R_hi, valid, levels=[0.5, 1.5], colors=['#9467bd'], alpha=0.35)
ax.contour(R_lo, R_hi, (i_fb_hold-(i_sink+ib_demand)), levels=[0], colors='C0')
ax.contour(R_lo, R_hi, (i_sink_reset-i_src_reset),     levels=[0], colors='C3', linestyles='--')
ax.plot(200e3, 30e6, 'k*', ms=14, label='as-built (R5,R6)')
ax.set(xscale='log', yscale='log',
       xlabel=r'$R_{fb,lo}$ (R5) [$\Omega$]', ylabel=r'$R_{fb,hi}$ (R6) [$\Omega$]',
       title='Feedback — hold (blue, above) vs reset (red --, below), valid = purple')
ax.grid(True, which='both', alpha=0.2); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()


In [ ]:
# ---- Latch-core gain vs power: R_core_b / R_core_a (cleaned from cell 10) ----
# c_gain  : Q4 stays saturated -> Ic/Ib < BETA_FORCED
# c_power : core standing current within its own allocated budget
# CORRECTION: the original charged the EN divider's draw against the core budget. Since the
# EN divider must be re-spec'd anyway (section 2), budget the core on its own allocation.
P_LATCH_BUDGET = 1000e-6                                 # explicit latch-core allocation
P_EN_DIV       = VIN_PD_MAX**2 / (R_uv_top + R_uv_bot)   # reported for context only

r = np.logspace(3, 6, 500)
R_b, R_a = np.meshgrid(r, r)          # R_b=R_core_b (R7, Q4 drive), R_a=R_core_a (R8, feedback)
ic_q4 = (VOUT_NOM - VBE_ON - VCE_SAT) / R_b
ib_q4 = (VOUT_NOM - VBE_ON) / R_a
c_gain  = (ic_q4 / ib_q4) < BETA_FORCED
p_core  = VOUT_NOM**2 / R_b + VOUT_NOM**2 / R_a
c_power = p_core < P_LATCH_BUDGET
valid   = c_gain & c_power

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(R_b, R_a, valid, levels=[0.5, 1.5], colors=['#ff7f0e'], alpha=0.30)
ax.contour(R_b, R_a, ic_q4/ib_q4, levels=[BETA_FORCED], colors='C0')
ax.contour(R_b, R_a, p_core,      levels=[P_LATCH_BUDGET], colors='C3', linestyles='--')
ax.plot(3e6, 300e3, 'k*', ms=14, label='as-built (R7,R8)')
ax.set(xscale='log', yscale='log',
       xlabel=r'$R_{core,b}$ (R7) [$\Omega$]', ylabel=r'$R_{core,a}$ (R8) [$\Omega$]',
       title='Latch core — gain (blue) vs power (red --), valid = orange')
ax.grid(True, which='both', alpha=0.2); ax.legend(loc='lower right')
print(f"EN-divider draw @22V (context): {P_EN_DIV*1e6:.0f} uW | latch-core budget: {P_LATCH_BUDGET*1e6:.0f} uW")
plt.tight_layout(); plt.show()


## 5. Summary

**Verified (independent of latch topology):**
* The original `constants` references (`V_EN_OP_TYP`, `I_EN_STBY_NOM`, `I_HYS_TYP`) do not exist
  — use `V_EN_NOM`, `I_STBY_NOM`, `I_HYS_NOM`.
* As-built EN/UVLO divider **300 k / 100 k**: $V_{rise}=4.88$ V, $V_{fall}=3.94$ V,
  draw ≈ 1.27 mW @ 22 V.
  * **FAIL** turn-on ceiling (4.88 V > 4.5 V `V_USB_MIN`) — the part won't be enabled at a sagging USB 4.5 V.
  * **FAIL** power budget (1.27 mW > 500 µW).
  * To centre $V_{rise}$ at ~4.15 V, target ratio $R_{top}/R_{bot}\approx2.4$ (e.g. 240 k / 100 k).
* The divider is **over-constrained**: tight hysteresis and the 500 µW budget cannot both be met
  at the LM5176's $I_{HYS}$ — which is the correct justification for the external latch.

**Corrections applied to the region search (§4) vs the original cells 6–10:**
* `IC_LM5176` attribute names fixed; cells 6/7 referenced `V_USB_MAX` / `vce_sat_max` before
  definition (NameError) — removed.
* The 6-level nested loop in the original cell 7 ($18^6\approx34$M iterations) is replaced with
  vectorised `meshgrid` sweeps.
* `PWR_LIMIT` was `200e-3` (200 mW) with a `1000uW` comment — a 200× unit error.
* Q2 "must hold the latch off" is a normal-operation condition → evaluated at the operating
  minimum (5 V), not 3.8 V (the original windows overlapped and had no solution).
* The RESET inequality is a low-VOUT event → source term evaluated at `V_OUT_RELEASE`, not
  `VOUT_NOM`.
* `plt.contour(..., boolean_array, levels=[0])` calls were dropped (degenerate).

**To confirm in LTspice before committing the latch resistors (§4):**
* Q2/Q3/Q4 pin assignments and the exact node each of R5–R8 lands on.
* The SET margin: `VIN_set` (Q2 release) must stay ≥ `HYS_MIN_GAP` above `V_fall` across VBE/temperature.
* The HOLD and RESET inequalities, against a `.tran` sweep using the V1/V2 PWL stimulus already
  in the schematic.
